<a href="https://colab.research.google.com/github/Pushkar1-GitHub/AI-in-Biology/blob/main/Copy_of_LSTM_based_EHR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

# 1. Hyperparameters
INPUT_DIM = 50   # Number of unique medical codes (ICD, meds, etc.)
EMBED_DIM = 32   # Dimension of the medical embeddings
HIDDEN_DIM = 64  # LSTM hidden units
OUTPUT_DIM = 1   # Binary classification (e.g., Risk of Readmission: 0 or 1)
SEQ_LENGTH = 10  # Number of past visits to look at

# 2. Define the Deep Learning Model (LSTM-based EHR)
class EHR_LSTM(nn.Module):
    def __init__(self, input_dim, embed_dim, hidden_dim):
        super(EHR_LSTM, self).__init__()
        # Embedding layer converts sparse codes into dense vectors
        self.embedding = nn.Embedding(input_dim, embed_dim)

        # LSTM processes the temporal sequence of visits
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)

        # Fully connected layer for the final prediction
        self.fc = nn.Linear(hidden_dim, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        # x shape: (batch_size, seq_length)
        x = self.embedding(x)             # Output: (batch_size, seq_len, embed_dim)

        # lstm_out: (batch_size, seq_len, hidden_dim)
        # hidden: (1, batch_size, hidden_dim) - the final state
        lstm_out, (hidden, cell) = self.lstm(x)

        # We take the last hidden state to represent the entire patient history
        last_state = hidden[-1]
        out = self.fc(last_state)
        return self.sigmoid(out)

# 3. Initialize Model, Loss, and Optimizer
model = EHR_LSTM(INPUT_DIM, EMBED_DIM, HIDDEN_DIM)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# 4. Generate Synthetic Data (Example for Classroom)
# 100 patients, each with 10 visits. Codes are integers between 0-49.
dummy_input = torch.randint(0, INPUT_DIM, (100, SEQ_LENGTH))
dummy_labels = torch.randint(0, 2, (100, 1)).float()

# 5. Simple Training Loop
print("Starting Training...")
for epoch in range(5):
    optimizer.zero_grad()
    outputs = model(dummy_input)
    loss = criterion(outputs, dummy_labels)
    loss.backward()
    optimizer.step()
    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

print("\nModel trained. Ready for risk prediction.")

Starting Training...
Epoch 1, Loss: 0.6915
Epoch 2, Loss: 0.6881
Epoch 3, Loss: 0.6847
Epoch 4, Loss: 0.6813
Epoch 5, Loss: 0.6779

Model trained. Ready for risk prediction.
